https://api.usaspending.gov/docs/endpoints

specific url documentation:https://github.com/fedspendingtransparency/usaspending-api/blob/master/usaspending_api/api_contracts/contracts/v2/search/spending_by_award.md

In [1]:
import os
import time
import requests
import pandas as pd
from datetime import date
from dotenv import load_dotenv

load_dotenv() 
print ("Libraries imported")

Libraries imported


In [2]:
from datetime import date, timedelta

yesterday = date.today() - timedelta(days=1)

yesterday_str = yesterday.isoformat()

print(yesterday_str)

2026-09-21


In [3]:
ByAward = requests.post(
    url = "https://api.usaspending.gov/api/v2/search/spending_by_award/",
    json = { 
        "subawards": False,
        "limit": 10,
        "page": 1,
        "filters": {
            "award_type_codes": ["A", "B", "C", "D"],
            "psc_codes": ["1550"],
            "time_period": [{"start_date": "2025-10-01", "end_date": yesterday_str}]
        },
        "fields" : [
            "generated_internal_id",
            "Award ID",
            "Recipient Name",
            "Awarding Agency",
            "Funding Agency",
            "Description",
            "Base Obligation Date",
            "Last Modified Date",
            "Start Date",
            "End Date",
            "Award Amount",
            "Contract Award Type",
            "NAICS",
            "Total Outlays"
        ]
    }
)

In [4]:
print(ByAward.status_code)

print(ByAward.json())

200
{'spending_level': 'awards', 'limit': 10, 'results': [{'internal_id': 361668682, 'generated_internal_id': 'CONT_AWD_W91ZRS26FC010_9700_47QSMS25D008P_4732', 'Award ID': 'W91ZRS26FC010', 'Recipient Name': 'W S DARLEY & CO', 'Awarding Agency': 'Department of Defense', 'Funding Agency': 'Department of Defense', 'Description': 'DRONE RTI', 'Base Obligation Date': '2026-05-22', 'Last Modified Date': '2026-08-07 07:03:05', 'Start Date': '2026-05-22', 'End Date': '2026-05-22', 'Award Amount': 23701.2, 'Contract Award Type': 'DELIVERY ORDER', 'NAICS': {'code': '339113', 'description': 'SURGICAL APPLIANCE AND SUPPLIES MANUFACTURING'}, 'Total Outlays': None, 'awarding_agency_id': 1173, 'agency_slug': 'department-of-defense'}, {'internal_id': 360126450, 'generated_internal_id': 'CONT_AWD_W91QVN26PA017_9700_-NONE-_-NONE-', 'Award ID': 'W91QVN26PA017', 'Recipient Name': 'KENCOA AEROSPACE CORPORATION', 'Awarding Agency': 'Department of Defense', 'Funding Agency': 'Department of Defense', 'Descrip

In [5]:
print(len(ByAward.json()["results"]))

10


https://github.com/fedspendingtransparency/usaspending-api/blob/master/usaspending_api/api_contracts/contracts/v2/search/spending_by_award_count.md

In [6]:
HowMuch = requests.post(
    url = "https://api.usaspending.gov/api/v2/search/spending_by_award_count/",
    json = { 
        "subawards": False,
        "limit": 10,
        "page": 1,
        "filters": {
            "award_type_codes": ["A", "B", "C", "D"],
            "psc_codes": ["1550"],
            "time_period": [{"start_date": "2025-10-01", "end_date": yesterday_str}]
        },
        "fields" : [
            "generated_internal_id",
            "Award ID",
            "Recipient Name",
            "Awarding Agency",
            "Funding Agency",
            "Description",
            "Base Obligation Date",
            "Last Modified Date",
            "Start Date",
            "End Date",
            "Award Amount",
            "Contract Award Type",
            "NAICS",
            "Total Outlays"
        ]
    }
)

In [7]:
print(HowMuch.json())

{'results': {'contracts': 574, 'direct_payments': 0, 'grants': 0, 'idvs': 0, 'loans': 0, 'other': 0}, 'spending_level': 'awards', 'messages': ['For searches, time period start and end dates are currently limited to an earliest date of 2007-10-01.  For data going back to 2000-10-01, use either the Custom Award Download feature on the website or one of our download or bulk_download API endpoints as listed on https://api.usaspending.gov/docs/endpoints. ']}


In [8]:
contracts = []

page_number = 1

ifNextPage = True

while ifNextPage:
    response = requests.post(
        url = "https://api.usaspending.gov/api/v2/search/spending_by_award/",
            json = { 
        "subawards": False,
        "limit": 100,
        "page": page_number,
        "filters": {
            "award_type_codes": ["A", "B", "C", "D"],
            "psc_codes": ["1550"],
            "time_period": [{"start_date": "2025-10-01", "end_date": yesterday_str}]
        },
        "fields" : [
            "generated_internal_id",
            "Award ID",
            "Recipient Name",
            "Awarding Agency",
            "Funding Agency",
            "Description",
            "Base Obligation Date",
            "Last Modified Date",
            "Start Date",
            "End Date",
            "Award Amount",
            "Contract Award Type",
            "NAICS",
            "Total Outlays"
        ]
    }
    )
    ifNextPage = response.json()["page_metadata"]["hasNext"]
    page_number = page_number + 1
    contracts.extend(response.json()["results"])
    

In [9]:
len(contracts)

574

In [10]:
print(contracts[0])

{'internal_id': 361668682, 'generated_internal_id': 'CONT_AWD_W91ZRS26FC010_9700_47QSMS25D008P_4732', 'Award ID': 'W91ZRS26FC010', 'Recipient Name': 'W S DARLEY & CO', 'Awarding Agency': 'Department of Defense', 'Funding Agency': 'Department of Defense', 'Description': 'DRONE RTI', 'Base Obligation Date': '2026-05-22', 'Last Modified Date': '2026-08-07 07:03:05', 'Start Date': '2026-05-22', 'End Date': '2026-05-22', 'Award Amount': 23701.2, 'Contract Award Type': 'DELIVERY ORDER', 'NAICS': {'code': '339113', 'description': 'SURGICAL APPLIANCE AND SUPPLIES MANUFACTURING'}, 'Total Outlays': None, 'awarding_agency_id': 1173, 'agency_slug': 'department-of-defense'}


In [11]:
import sqlite3 as sql
from datetime import date

In [12]:
connection = sql.connect("../UAS_Tracker.db")

In [13]:
sql_string = """INSERT OR REPLACE INTO contracts 
                (api_internal_id,
                award_id,
                recipient,
                awarding_agency,
                amount_usd,
                psc_code,
                description,
                award_base_date,
                naics_code,
                last_modified_date,
                raw_json)

                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                 """

In [14]:
import json

In [15]:
cursor = connection.cursor() 

for contract in contracts:
    naics = contract.get("NAICS")
    if naics:
        naics_code = naics["code"]
    else: 
        naics_code = None
    cursor.execute( sql_string, (
        contract["generated_internal_id"],
        contract["Award ID"],
        contract["Recipient Name"],
        contract["Awarding Agency"],
        contract["Award Amount"],
        "1550",
        contract["Description"],
        contract["Base Obligation Date"],
        naics_code,
        contract["Last Modified Date"],
        json.dumps(contract)
        )
        )

connection.commit()

In [16]:
cursor.execute("""SELECT COUNT(*) FROM contracts""")

print(cursor.fetchall())

[(518,)]


In [17]:
SQL_CSV = pd.read_sql_query("""SELECT api_internal_id, 
                             award_id,
                             recipient, 
                             awarding_agency, 
                             amount_usd,  
                             description, 
                             award_base_date,
                             last_modified_date
                             FROM contracts ORDER BY api_internal_id""", connection)

SQL_CSV.to_csv(f"../Exported_SQL_Contracts_Table.csv", index=False)